# Overview
This notebook is intended to recover results from Coetzer and Haines (2017), using the reparameterization methods described in this thesis rather than the barycentric coordinate approach used by the authors.

In [2]:
include("../../src/MetaDoE.jl")
using .MetaDoE: Experiments, ConstraintEnforcement, Constraints, PSO, Objectives, Designs, Models, HitAndRun, TensorOps
using Base.Iterators
using LinearAlgebra
using NPZ
using HiGHS
using Polyhedra
using Random

# 3D Mixture Example

In [3]:
# Specify number of trials and factors
K = 3
N = 15

# Define constraints
A = [
    0.0  -1.0   0.0;
    0.0   0.0   1.0;
    5.0   4.0   0.0;
   -20.0   5.0   0.0
]

b = [
   -1/10;
    3/5;
   39/10;
   -3
]

# Apply simplex constraints
A, b = Constraints.simplex(A, b)

# Define experiment
model = Models.scheffe(3)
experiment = Experiments.create(N, K-1, model)
experiment = Experiments.with_linear_constraints(experiment, A, b)

Main.MetaDoE.Experiments.Experiment(Dict("1" => 1, "2" => 2), Main.MetaDoE.ConstraintEnforcement.LinearConstraints([0.0 -1.0; -1.0 -1.0; … ; -0.0 -1.0; 1.0 1.0], [0.2333333333333333, 0.26666666666666666, 0.8999999999999999, 2.0, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333]), Main.MetaDoE.Models.var"#model_builder#19"{Int64, Int64, Bool, Vector{Any}, Bool, Bool}(1, 2, false, Any[], false, true), 15, 2)

In [4]:
function pso_optimize()
    # Create optimization context
    objective = Objectives.D ∘ model ∘ Constraints.psi
    params = PSO.create_hyperparams(500)
    context = PSO.create_context(experiment, objective; hyperparams=params, use_model=false, callback=identity)

    # Optimize
    runner_state, history = PSO.optimize(context)

    # Extract optimizer
    return PSO.get_optimizer(runner_state)
end

optims_15 = []
for i in 1:100
    print("Trial $(i)")
    result = pso_optimize()
    push!(optims_15, result)
end

Trial 1Trial 2Trial 3Trial 4Trial 5Trial 6Trial 7Trial 8Trial 9Trial 10Trial 11Trial 12Trial 13Trial 14Trial 15Trial 16Trial 17Trial 18Trial 19Trial 20Trial 21Trial 22Trial 23Trial 24Trial 25Trial 26Trial 27Trial 28Trial 29Trial 30Trial 31Trial 32Trial 33Trial 34Trial 35Trial 36Trial 37Trial 38Trial 39Trial 40Trial 41Trial 42Trial 43Trial 44Trial 45Trial 46Trial 47Trial 48Trial 49Trial 50Trial 51Trial 52Trial 53Trial 54Trial 55Trial 56Trial 57Trial 58Trial 59Trial 60Trial 61Trial 62Trial 63Trial 64Trial 65Trial 66Trial 67Trial 68Trial 69Trial 70Trial 71Trial 72Trial 73Trial 74Trial 75Trial 76Trial 77Trial 78Trial 79Trial 80Trial 81Trial 82Trial 83Trial 84Trial 85Trial 86Trial 87Trial 88Trial 89Trial 90Trial 91Trial 92Trial 93Trial 94Trial 95Trial 96Trial 97Trial 98Trial 99Trial 100

## 12 Design Points

In [93]:
function D_trad(X)
    det(X' * X)
end

scores = map(D_trad ∘ model ∘ Constraints.little_psi, [o[1] for o in optims_12])
best_idx = argmin(scores)
worst_idx = argmax(scores)
scores[best_idx], scores[worst_idx]

(2.112266516995832e-13, 5.566665624330127e-13)

In [95]:
X = [
    .7 .3 .2 .3 .5902 .4098 .2702 .2279 .4118 .5738 .4211 .3360;
    .1 .6 .2 .1 .2372 .4628 .4808 .3117 .1 .1 .2911 .2264;
    .2 .1 .6 .6 .1725 .1274 .2490 .4604 .4882 .3263 .2878 .4376
]'

F = model(X)
coetzer_score = det(F' * F)

5.649418538016769e-13

In [100]:
(scores[best_idx] / coetzer_score)^(1/7)

0.868887533778017

In [99]:
(scores[worst_idx] / coetzer_score)^(1/7)

0.9978941657112467

## 15 Points

In [5]:
scores = map(D_trad ∘ model ∘ Constraints.little_psi, [o[1] for o in optims_15])
best_idx = argmin(scores)
worst_idx = argmax(scores)
scores[best_idx], scores[worst_idx]

LoadError: UndefVarError: `D_trad` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
X_15 = [.7 .3 .3 .2 .3 .5902 .5902 .4098 .4098 .2702 .2279 .4118 .5738 .4211 .3360;
.1 .6 .6 .2 .1 .2373 .2373 .4628 .4628 .4808 .3117 .100 .100 .2911 .2264;
.2 .1 .1 .6 .6 .1725 .1725 .1274 .1274 .2490 .4604 .4882 .3262 .2878 .4376]'

15×3 adjoint(::Matrix{Float64}) with eltype Float64:
 0.7     0.1     0.2
 0.3     0.6     0.1
 0.3     0.6     0.1
 0.2     0.2     0.6
 0.3     0.1     0.6
 0.5902  0.2373  0.1725
 0.5902  0.2373  0.1725
 0.4098  0.4628  0.1274
 0.4098  0.4628  0.1274
 0.2702  0.4808  0.249
 0.2279  0.3117  0.4604
 0.4118  0.1     0.4882
 0.5738  0.1     0.3262
 0.4211  0.2911  0.2878
 0.336   0.2264  0.4376

In [ ]:
F = model(X_15)
coetzer_score_15 = det(F' * F)

2.4064358975286477e-12

In [ ]:
(scores[worst_idx] / coetzer_score_15)^(1/7)

0.9768158964190686

In [ ]:
(scores[best_idx] / coetzer_score_15)^(1/7)

0.8457378023995097

# 4D Mixture Example

In [6]:
N = 12
K = 4

A = [
    -1.0  0.0  0.0  0.0;
     1.0  0.0  0.0  0.0;
     0.0 -1.0  0.0  0.0;
     0.0  1.0  0.0  0.0;
     0.0  0.0 -1.0  0.0;
     0.0  0.0  1.0  0.0;
     0.0  0.0  0.0 -1.0;
     0.0  0.0  0.0  1.0
]

b = [
    -0.2;
     0.65;
    -0.1;
     0.55;
    -0.1;
     0.2;
    -0.15;
     0.35
]

A, b = Constraints.simplex(A, b)

model = Models.scheffe(2)
experiment = Experiments.create(N, K-1, model)
experiment = Experiments.with_linear_constraints(experiment, A, b)

Main.MetaDoE.Experiments.Experiment(Dict("1" => 1, "2" => 2, "3" => 3), Main.MetaDoE.ConstraintEnforcement.LinearConstraints([-1.0 0.0 0.0; 1.0 0.0 0.0; … ; -0.0 -0.0 -1.0; 1.0 1.0 1.0], [0.04999999999999999, 0.4, 0.15, 0.30000000000000004, 0.15, -0.04999999999999999, 0.1, 0.09999999999999998, 0.25, 0.25, 0.25, 0.25]), Main.MetaDoE.Models.var"#model_builder#19"{Int64, Int64, Bool, Vector{Any}, Bool, Bool}(1, 1, false, Any[], false, true), 12, 3)

In [ ]:
context = PSO.create_context(experiment, Objectives.D)
runner_state, history = PSO.optimize(context)
optimizer, optimum = PSO.get_optimizer(runner_state)

# Save samples
sampler = HitAndRun.hit_and_run(experiment.constraints.A, experiment.constraints.b)
samples = sampler(10000)
npzwrite("../data/coetzer_3d_samples.npy", samples)

# Save optimizer
npzwrite("../data/coetzer_3d_12.npy", optimizer)

# Save vertices
Constraints.save_vertices(experiment; location = "../data/coetzer_verts_3d.npy")

Iteration: 0 Best score: 16.054887458177845
Iteration: 1 Best score: 15.819263791014798
Iteration: 2 Best score: 15.110657077433935
Iteration: 3 Best score: 15.110657077433935
Iteration: 4 Best score: 15.110657077433935
Iteration: 5 Best score: 15.110657077433935
Iteration: 6 Best score: 15.110657077433935
Iteration: 7 Best score: 15.110657077433935
Iteration: 8 Best score: 15.110657077433935
Iteration: 9 Best score: 15.110657077433935
Iteration: 10 Best score: 15.110657077433935
Iteration: 11 Best score: 15.110657077433935
Iteration: 12 Best score: 15.110657077433935
Iteration: 13 Best score: 13.283278248735641
Iteration: 14 Best score: 10.232193171629696
Iteration: 15 Best score: 8.110847054264521
Iteration: 16 Best score: 8.110847054264521
Iteration: 17 Best score: 8.110847054264521
Iteration: 18 Best score: 8.110847054264521
Iteration: 19 Best score: 8.110847054264521
Iteration: 20 Best score: 8.110847054264521
Iteration: 21 Best score: 8.110847054264521
Iteration: 22 Best score: 8

([-0.04219641150412927 0.015841043029974546 -0.04999999999999999; -0.010066115427273205 0.09303805368938839 -0.04999999999999999; … ; 0.1254198908953493 0.02439512105694945 -0.04999999999999999; 0.04827823698003192 -0.041062570440394866 -0.04999999999999999], 8.041085284368748)